# 01 - Dataset Familiarization

This notebook contains initial exploration and notes for the Amsterdam dataset.

In [48]:
from pathlib import Path

RAW_DATA_PATH = Path("../data/raw/amsterdam/2026-06-15")

expected_files = [
    "listings.csv.gz",
    "calendar.csv.gz",
    "reviews.csv.gz",
    "listings.csv",
    "reviews.csv",
    "neighbourhoods.csv",
    "neighbourhoods.geojson"
]

for file_name in expected_files:
    file_path = RAW_DATA_PATH / file_name
    print(file_name, "✅ Found" if file_path.exists() else "❌ Missing")

listings.csv.gz ✅ Found
calendar.csv.gz ✅ Found
reviews.csv.gz ✅ Found
listings.csv ✅ Found
reviews.csv ✅ Found
neighbourhoods.csv ✅ Found
neighbourhoods.geojson ✅ Found


In [49]:
import pandas as pd
from pathlib import Path

RAW_DATA_PATH = Path("../data/raw/amsterdam/2026-06-15")

files = {
    "detailed_listings": RAW_DATA_PATH / "listings.csv.gz",
    "calendar": RAW_DATA_PATH / "calendar.csv.gz",
    "detailed_reviews": RAW_DATA_PATH / "reviews.csv.gz",
    "summary_listings": RAW_DATA_PATH / "listings.csv",
    "summary_reviews": RAW_DATA_PATH / "reviews.csv",
    "neighbourhoods": RAW_DATA_PATH / "neighbourhoods.csv",
    "neighbourhoods_geojson": RAW_DATA_PATH / "neighbourhoods.geojson"
}

for name, path in files.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'} - {path}")

detailed_listings: FOUND - ..\data\raw\amsterdam\2026-06-15\listings.csv.gz
calendar: FOUND - ..\data\raw\amsterdam\2026-06-15\calendar.csv.gz
detailed_reviews: FOUND - ..\data\raw\amsterdam\2026-06-15\reviews.csv.gz
summary_listings: FOUND - ..\data\raw\amsterdam\2026-06-15\listings.csv
summary_reviews: FOUND - ..\data\raw\amsterdam\2026-06-15\reviews.csv
neighbourhoods: FOUND - ..\data\raw\amsterdam\2026-06-15\neighbourhoods.csv
neighbourhoods_geojson: FOUND - ..\data\raw\amsterdam\2026-06-15\neighbourhoods.geojson


In [50]:
datasets = {}

datasets["detailed_listings"] = pd.read_csv(files["detailed_listings"])
datasets["calendar"] = pd.read_csv(files["calendar"])
datasets["detailed_reviews"] = pd.read_csv(files["detailed_reviews"])
datasets["summary_listings"] = pd.read_csv(files["summary_listings"])
datasets["summary_reviews"] = pd.read_csv(files["summary_reviews"])
datasets["neighbourhoods"] = pd.read_csv(files["neighbourhoods"])

for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print(df.head(3))

detailed_listings
Rows: 10369
Columns: 90
      id                         listing_url       scrape_id last_scraped  \
0  28871  https://www.airbnb.com/rooms/28871  20260615212022   2026-06-16   
1  29051  https://www.airbnb.com/rooms/29051  20260615212022   2026-06-16   
2  44129  https://www.airbnb.com/rooms/44129  20260615212022   2026-06-24   

            source                              name  \
0  previous scrape           Comfortable double room   
1  previous scrape  Comfortable single / double room   
2  previous scrape     Luxury design with canal view   

                                         description  neighborhood_overview  \
0          Basic bedroom in the center of Amsterdam.                    NaN   
1  This room can also be rented as a single or a ...                    NaN   
2  Welcome to my little gem<br /><br />Cozy, brig...                    NaN   

                                         picture_url  host_id  ...  \
0  https://a0.muscache.com/pictures/1

## 1. Dataset Inventory

This section summarizes all raw Amsterdam datasets loaded from Inside Airbnb.  
The goal is to understand the size and structure of each file before cleaning or analysis.

In [51]:
import pandas as pd
from pathlib import Path

RAW_DATA_PATH = Path("../data/raw/amsterdam/2026-06-15")

files = {
    "detailed_listings": RAW_DATA_PATH / "listings.csv.gz",
    "calendar": RAW_DATA_PATH / "calendar.csv.gz",
    "detailed_reviews": RAW_DATA_PATH / "reviews.csv.gz",
    "summary_listings": RAW_DATA_PATH / "listings.csv",
    "summary_reviews": RAW_DATA_PATH / "reviews.csv",
    "neighbourhoods": RAW_DATA_PATH / "neighbourhoods.csv"
}

datasets = {
    "detailed_listings": pd.read_csv(files["detailed_listings"]),
    "calendar": pd.read_csv(files["calendar"]),
    "detailed_reviews": pd.read_csv(files["detailed_reviews"]),
    "summary_listings": pd.read_csv(files["summary_listings"]),
    "summary_reviews": pd.read_csv(files["summary_reviews"]),
    "neighbourhoods": pd.read_csv(files["neighbourhoods"])
}

inventory = []

for name, df in datasets.items():
    inventory.append({
        "dataset_name": name,
        "file_name": files[name].name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    })

inventory_df = pd.DataFrame(inventory)
inventory_df

,dataset_name,file_name,rows,columns,memory_mb
0,detailed_listings,listings.csv.gz,10369,90,28.00
1,calendar,calendar.csv.gz,3819725,5,185.78
2,detailed_reviews,reviews.csv.gz,545162,6,164.49
3,summary_listings,listings.csv,10465,19,2.57
4,summary_reviews,reviews.csv,545162,2,13.52
5,neighbourhoods,neighbourhoods.csv,22,2,0.00


In [52]:
REPORTS_PATH = Path("../reports/data_quality")
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

inventory_df.to_csv(REPORTS_PATH / "amsterdam_dataset_inventory.csv", index=False)

print("Dataset inventory saved successfully.")

Dataset inventory saved successfully.


### Initial Observation

The Amsterdam dataset contains multiple raw files at different levels of detail.  
The detailed listings file contains rich listing, host, pricing, location, and review score attributes.  
The calendar file provides daily availability and pricing records, while the reviews file provides review-level data.  
The neighbourhood file supports geographic grouping and filtering.

This confirms that the dataset can support data engineering, exploratory analysis, statistical testing, machine learning, and NLP-based review analysis.

## 2. Schema Profiling

This section creates a column-level schema profile for each raw dataset.  
For every column, the profile records data type, missing values, uniqueness, and sample values.  
This helps identify columns that require cleaning, type conversion, validation, or special interpretation.

In [53]:
def create_schema_profile(df, dataset_name):
    """
    Create a column-level schema profile for a dataframe.
    """
    profile_rows = []

    for column in df.columns:
        sample_values = (
            df[column]
            .dropna()
            .astype(str)
            .unique()[:5]
        )

        profile_rows.append({
            "dataset_name": dataset_name,
            "column_name": column,
            "data_type": str(df[column].dtype),
            "non_null_count": df[column].notna().sum(),
            "null_count": df[column].isna().sum(),
            "null_percentage": round((df[column].isna().sum() / len(df)) * 100, 2),
            "unique_count": df[column].nunique(dropna=True),
            "sample_values": " | ".join(sample_values)
        })

    return pd.DataFrame(profile_rows)

In [54]:
schema_profiles = []

for dataset_name, df in datasets.items():
    schema_profile = create_schema_profile(df, dataset_name)
    schema_profiles.append(schema_profile)

all_schema_profiles_df = pd.concat(schema_profiles, ignore_index=True)

all_schema_profiles_df.head(20)

,dataset_name,column_name,data_type,non_null_count,null_count,null_percentage,unique_count,sample_values
0,detailed_listings,id,int64,10369,0,0.00,10369,28871 | 29051 | 44129 | 44391 | 48373
1,detailed_listings,listing_url,str,10369,0,0.00,10369,https://www.airbnb.com/rooms/28871 | https://w...
2,detailed_listings,scrape_id,int64,10369,0,0.00,1,20260615212022
3,detailed_listings,last_scraped,str,10369,0,0.00,3,2026-06-16 | 2026-06-24 | 2026-06-15
4,detailed_listings,source,str,10369,0,0.00,2,previous scrape | city scrape
5,detailed_listings,name,str,10369,0,0.00,10076,Comfortable double room | Comfortable single /...
6,detailed_listings,description,str,9919,450,4.34,9640,Basic bedroom in the center of Amsterdam. | Th...
7,detailed_listings,neighborhood_overview,float64,0,10369,100.00,0,
8,detailed_listings,picture_url,str,10369,0,0.00,10315,https://a0.muscache.com/pictures/160889/362340...
9,detailed_listings,host_id,int64,10369,0,0.00,9104,124245 | 187728 | 194779 | 220434 | 225987


In [55]:
schema_output_path = REPORTS_PATH / "amsterdam_schema_profile.csv"

all_schema_profiles_df.to_csv(schema_output_path, index=False)

print(f"Schema profile saved to: {schema_output_path}")

PermissionError: [Errno 13] Permission denied: '..\\reports\\data_quality\\amsterdam_schema_profile.csv'

## 3. Missing Values and Data Quality Analysis

This section evaluates the completeness and basic quality of each raw dataset.  
The goal is to identify missing values, duplicate records, key column issues, and domain validation problems before cleaning and transformation.

These checks help decide which columns can be trusted, which columns need cleaning, and which columns should be excluded from downstream analysis.

In [56]:
missing_value_rows = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        null_count = df[column].isna().sum()
        null_percentage = round((null_count / len(df)) * 100, 2)

        missing_value_rows.append({
            "dataset_name": dataset_name,
            "column_name": column,
            "total_rows": len(df),
            "null_count": null_count,
            "null_percentage": null_percentage
        })

missing_values_df = pd.DataFrame(missing_value_rows)

missing_values_df = missing_values_df.sort_values(
    by=["dataset_name", "null_percentage"],
    ascending=[True, False]
)

missing_values_df.head(30)

,dataset_name,column_name,total_rows,null_count,null_percentage
90,calendar,listing_id,3819725,0,0.00
91,calendar,date,3819725,0,0.00
92,calendar,available,3819725,0,0.00
93,calendar,minimum_nights,3819725,0,0.00
94,calendar,maximum_nights,3819725,0,0.00
7,detailed_listings,neighborhood_overview,10369,10369,100.00
14,detailed_listings,host_since,10369,10369,100.00
21,detailed_listings,host_response_time,10369,10369,100.00
22,detailed_listings,host_response_rate,10369,10369,100.00
23,detailed_listings,host_acceptance_rate,10369,10369,100.00


In [57]:
missing_output_path = REPORTS_PATH / "amsterdam_missing_values_summary.csv"

missing_values_df.to_csv(missing_output_path, index=False)

print(f"Missing values summary saved to: {missing_output_path}")

Missing values summary saved to: ..\reports\data_quality\amsterdam_missing_values_summary.csv


### Duplicate Record Check

This check identifies exact duplicate rows in each raw dataset.  
Exact duplicates can indicate repeated ingestion, scraping duplication, or source-level data quality issues.

In [58]:
duplicate_summary_rows = []

for dataset_name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    duplicate_percentage = round((duplicate_count / len(df)) * 100, 4)

    duplicate_summary_rows.append({
        "dataset_name": dataset_name,
        "total_rows": len(df),
        "duplicate_rows": duplicate_count,
        "duplicate_percentage": duplicate_percentage
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary_rows)
duplicate_summary_df

,dataset_name,total_rows,duplicate_rows,duplicate_percentage
0,detailed_listings,10369,0,0.0000
1,calendar,3819725,0,0.0000
2,detailed_reviews,545162,0,0.0000
3,summary_listings,10465,0,0.0000
4,summary_reviews,545162,22541,4.1347
5,neighbourhoods,22,0,0.0000


In [59]:
duplicate_output_path = REPORTS_PATH / "amsterdam_duplicate_summary.csv"

duplicate_summary_df.to_csv(duplicate_output_path, index=False)

print(f"Duplicate summary saved to: {duplicate_output_path}")

Duplicate summary saved to: ..\reports\data_quality\amsterdam_duplicate_summary.csv


### Key Column Quality Check

This check evaluates whether important ID columns are complete and unique.  
These columns are important because they are used to connect listings, calendar records, reviews, hosts, and neighbourhoods.

In [60]:
key_checks = [
    {
        "dataset_name": "detailed_listings",
        "key_column": "id",
        "expected_role": "Primary key for detailed listings"
    },
    {
        "dataset_name": "summary_listings",
        "key_column": "id",
        "expected_role": "Primary key for summary listings"
    },
    {
        "dataset_name": "calendar",
        "key_column": "listing_id",
        "expected_role": "Foreign key referencing listings"
    },
    {
        "dataset_name": "detailed_reviews",
        "key_column": "listing_id",
        "expected_role": "Foreign key referencing listings"
    },
    {
        "dataset_name": "summary_reviews",
        "key_column": "listing_id",
        "expected_role": "Foreign key referencing listings"
    },
    {
        "dataset_name": "detailed_listings",
        "key_column": "host_id",
        "expected_role": "Host identifier"
    }
]

key_quality_rows = []

for check in key_checks:
    dataset_name = check["dataset_name"]
    key_column = check["key_column"]
    df = datasets[dataset_name]

    if key_column in df.columns:
        key_quality_rows.append({
            "dataset_name": dataset_name,
            "key_column": key_column,
            "expected_role": check["expected_role"],
            "total_rows": len(df),
            "non_null_count": df[key_column].notna().sum(),
            "null_count": df[key_column].isna().sum(),
            "unique_count": df[key_column].nunique(dropna=True),
            "duplicate_key_count": df[key_column].duplicated().sum()
        })
    else:
        key_quality_rows.append({
            "dataset_name": dataset_name,
            "key_column": key_column,
            "expected_role": check["expected_role"],
            "total_rows": len(df),
            "non_null_count": None,
            "null_count": None,
            "unique_count": None,
            "duplicate_key_count": None
        })

key_quality_df = pd.DataFrame(key_quality_rows)
key_quality_df

,dataset_name,key_column,expected_role,total_rows,non_null_count,null_count,unique_count,duplicate_key_count
0,detailed_listings,id,Primary key for detailed listings,10369,10369,0,10369,0
1,summary_listings,id,Primary key for summary listings,10465,10465,0,10465,0
2,calendar,listing_id,Foreign key referencing listings,3819725,3819725,0,10465,3809260
3,detailed_reviews,listing_id,Foreign key referencing listings,545162,545162,0,9432,535730
4,summary_reviews,listing_id,Foreign key referencing listings,545162,545162,0,9432,535730
5,detailed_listings,host_id,Host identifier,10369,10369,0,9104,1265


In [61]:
key_quality_output_path = REPORTS_PATH / "amsterdam_key_quality_summary.csv"

key_quality_df.to_csv(key_quality_output_path, index=False)

print(f"Key quality summary saved to: {key_quality_output_path}")

Key quality summary saved to: ..\reports\data_quality\amsterdam_key_quality_summary.csv


### Domain Validation Checks

This section validates important fields against basic business rules.

Examples:
- Price should not be negative.
- Latitude should be between -90 and 90.
- Longitude should be between -180 and 180.
- Availability values should be valid.
- Review scores should fall within expected rating ranges.

In [62]:
def clean_price_value(value):
    """
    Convert price values like '$1,234.00' into numeric values.
    Returns NaN if the value cannot be converted.
    """
    if pd.isna(value):
        return pd.NA

    value = str(value)
    value = value.replace("$", "").replace(",", "").strip()

    try:
        return float(value)
    except ValueError:
        return pd.NA


validation_rows = []

# Detailed listings price validation
if "price" in datasets["detailed_listings"].columns:
    detailed_price = datasets["detailed_listings"]["price"].apply(clean_price_value)
    validation_rows.append({
        "dataset_name": "detailed_listings",
        "field": "price",
        "rule": "Price should be numeric and non-negative",
        "invalid_count": ((detailed_price.notna()) & (detailed_price < 0)).sum(),
        "missing_or_unparseable_count": detailed_price.isna().sum()
    })

# Summary listings price validation
if "price" in datasets["summary_listings"].columns:
    summary_price = pd.to_numeric(datasets["summary_listings"]["price"], errors="coerce")
    validation_rows.append({
        "dataset_name": "summary_listings",
        "field": "price",
        "rule": "Price should be numeric and non-negative",
        "invalid_count": ((summary_price.notna()) & (summary_price < 0)).sum(),
        "missing_or_unparseable_count": summary_price.isna().sum()
    })

# Calendar price validation
if "price" in datasets["calendar"].columns:
    calendar_price = datasets["calendar"]["price"].apply(clean_price_value)
    validation_rows.append({
        "dataset_name": "calendar",
        "field": "price",
        "rule": "Daily price should be numeric and non-negative",
        "invalid_count": ((calendar_price.notna()) & (calendar_price < 0)).sum(),
        "missing_or_unparseable_count": calendar_price.isna().sum()
    })

# Latitude validation
for dataset_name in ["detailed_listings", "summary_listings"]:
    if "latitude" in datasets[dataset_name].columns:
        latitude = pd.to_numeric(datasets[dataset_name]["latitude"], errors="coerce")
        validation_rows.append({
            "dataset_name": dataset_name,
            "field": "latitude",
            "rule": "Latitude should be between -90 and 90",
            "invalid_count": ((latitude.notna()) & ((latitude < -90) | (latitude > 90))).sum(),
            "missing_or_unparseable_count": latitude.isna().sum()
        })

# Longitude validation
for dataset_name in ["detailed_listings", "summary_listings"]:
    if "longitude" in datasets[dataset_name].columns:
        longitude = pd.to_numeric(datasets[dataset_name]["longitude"], errors="coerce")
        validation_rows.append({
            "dataset_name": dataset_name,
            "field": "longitude",
            "rule": "Longitude should be between -180 and 180",
            "invalid_count": ((longitude.notna()) & ((longitude < -180) | (longitude > 180))).sum(),
            "missing_or_unparseable_count": longitude.isna().sum()
        })

# Calendar availability validation
if "available" in datasets["calendar"].columns:
    valid_available_values = {"t", "f", True, False}
    available_values = datasets["calendar"]["available"]
    validation_rows.append({
        "dataset_name": "calendar",
        "field": "available",
        "rule": "Availability should be t/f or boolean",
        "invalid_count": (~available_values.isin(valid_available_values)).sum(),
        "missing_or_unparseable_count": available_values.isna().sum()
    })

# Review score validation
review_score_columns = [
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

for column in review_score_columns:
    if column in datasets["detailed_listings"].columns:
        score = pd.to_numeric(datasets["detailed_listings"][column], errors="coerce")
        validation_rows.append({
            "dataset_name": "detailed_listings",
            "field": column,
            "rule": "Review score should be between 0 and 5",
            "invalid_count": ((score.notna()) & ((score < 0) | (score > 5))).sum(),
            "missing_or_unparseable_count": score.isna().sum()
        })

domain_validation_df = pd.DataFrame(validation_rows)
domain_validation_df

,dataset_name,field,rule,invalid_count,missing_or_unparseable_count
0,detailed_listings,price,Price should be numeric and non-negative,0,3992
1,summary_listings,price,Price should be numeric and non-negative,0,3994
2,detailed_listings,latitude,Latitude should be between -90 and 90,0,0
3,summary_listings,latitude,Latitude should be between -90 and 90,0,0
4,detailed_listings,longitude,Longitude should be between -180 and 180,0,0
5,summary_listings,longitude,Longitude should be between -180 and 180,0,0
6,calendar,available,Availability should be t/f or boolean,0,0
7,detailed_listings,review_scores_rating,Review score should be between 0 and 5,0,1023
8,detailed_listings,review_scores_accuracy,Review score should be between 0 and 5,0,1023
9,detailed_listings,review_scores_cleanliness,Review score should be between 0 and 5,0,1024


In [63]:
domain_validation_output_path = REPORTS_PATH / "amsterdam_domain_validation_summary.csv"

domain_validation_df.to_csv(domain_validation_output_path, index=False)

print(f"Domain validation summary saved to: {domain_validation_output_path}")

Domain validation summary saved to: ..\reports\data_quality\amsterdam_domain_validation_summary.csv


### Data Quality Observations

Initial data quality profiling shows that the Amsterdam dataset is large enough for meaningful engineering, analytics, and machine learning work.  
The calendar dataset is the largest table because it stores daily availability and price records for each listing.  
The reviews dataset is also large and can support time-based demand analysis and NLP experiments.

Some columns contain high missing-value percentages, which indicates that not all raw fields are suitable for downstream analysis.  
The next stage will focus on selecting useful columns, cleaning price/date fields, validating key relationships, and preparing analytics-ready datasets.